# Cross-sectional feature selection — `unified_schema_all`

**The question changed.** The three notebooks before this one asked *will VCB go
up*, and [evaluation_study.ipynb](evaluation_study.ipynb) answered: not knowably —
five configurations, none clearing its own shuffled-label null, with `n_eff` and
not the feature pool as the binding constraint (CONTEXT.md §6b-6d).

This notebook asks the question §7 said to ask instead: **on a given day, which of
these stocks will beat the others?** That is a different label, a different metric
and a different panel — `unified_schema_all`, every ticker of
`silver.stocks_basic`, keyed and typed exactly like `unified_schema_vcb`.

| | single ticker | here |
|---|---|---|
| panel | 1 x 4,235 | **N x T**, up to 780 x 2,862 |
| target | `return_5day` | **`cs_rank_5day`** — the return's rank *within its date*, on [-0.5, +0.5] |
| features | raw levels / z-scored windows | **within-date ranks**, then windowed |
| metric | one pooled Spearman per fold | **one Spearman per DATE**, averaged |
| CV | expanding blocks of ROWS | expanding blocks of **DATES** |
| `n_eff` | `n_rows / h` | **`n_dates / h`** — width buys precision, not independence |

Everything else is the same package: the same six rankers, the same rank-average
ensemble, the same correlation prune, the same purge arithmetic `d + h - 1`, and
the same rule that a selection is worth nothing until it beats its own null.
`cross_sectional.CrossSectionalSelector` overrides six hooks on `FeatureSelector`
and inherits the rest, so the two studies are comparable by construction.

> ⚠️ **Read [cross_sectional.py](cross_sectional.py) §1 before changing anything
> here.** Three specific mistakes manufacture a cross-sectional result — splitting
> the CV by row, windowing down the stacked frame, and reporting a pooled IC — and
> all three produce numbers that look better than the ones below.

## Imports & theme

In [ ]:
import csv
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from feature_selection import cross_sectional as cs
from feature_selection import evaluation, plots

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plots.use_theme()

## Parameters

⚠️ `MIN_TRAIN` and the folds count **SESSIONS, not rows**. On a 100-name panel a
`min_train` of 600 rows would be six trading days.

⚠️ `N_NULL` is 10 here so the notebook finishes in a sitting. CONTEXT.md §9b
quotes the **20-draw** run; the verdict is the same, which is itself the point —
the result is not close enough to the bar for the draw count to decide it.

In [ ]:
UNIVERSE_CSV = "../../vn100.csv"   # the tradeable cross-section; None reads all 780
START, END = "2015-01-01", "2026-06-26"

LOOKBACK = 20         # d — window length, in sessions
HORIZON = 5           # h — label horizon, in sessions
MIN_WIDTH = 20        # drop dates with a narrower cross-section than this
MAX_FEATURES = 12     # cap on kept CHANNELS, after the redundancy prune
CORR_THRESHOLD = 0.9
N_SPLITS, MIN_TRAIN = 5, 600        # ⚠️ SESSIONS
HOLDOUT_START = "2024-06-01"
N_NULL = 10
DEVICE = "cpu"        # ⚠️ see the note on CUDA at the end of this notebook

TARGET = f"cs_rank_{HORIZON}day"

with open(UNIVERSE_CSV, encoding="utf-8-sig") as fh:
    TICKERS = sorted({row["ticker"].strip() for row in csv.DictReader(fh)})
len(TICKERS), TICKERS[:8]

## Read the panel

`read_universe_panel` filters **in SQL** — the full pool is 2.39 M rows and reading
it to keep 100 tickers costs ~1.5 GB to throw most of it away — then adds
`cs_rank_{h}day` per date.

⚠️ **`VNX` is excluded by default and this is not cosmetic.** Its silver
`close_adjust` is *negative* for 968 sessions, which makes `return_5day` reach
**-781**. That is a data defect, not a return, and it is the worst outlier in
`unified_schema_all`.

In [ ]:
panel = cs.read_universe_panel(
    tickers=TICKERS, start=START, end=END, min_width=MIN_WIDTH
)
cs.panel_summary(panel, TARGET).to_frame().T

## Why the target is a rank

`return_5day` on the whole universe spans **-781 to +4.2**. `cs_rank_5day` is
uniform on `[-0.5, +0.5]` by construction, on every date, whatever the market did
that week. Three problems disappear at once: the market factor, the outliers, and
the non-stationarity that made a price level act as a date proxy in §6c.

In [ ]:
comparison = pd.DataFrame({
    "return_5day": panel["return_5day"].describe(),
    "cs_rank_5day": panel[TARGET].describe(),
}).round(4)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
panel["return_5day"].clip(-0.3, 0.3).hist(bins=80, ax=axes[0])
axes[0].set_title("return_5day (clipped to ±0.30)")
panel[TARGET].hist(bins=80, ax=axes[1])
axes[1].set_title("cs_rank_5day — uniform on [-0.5, +0.5], every date")
for ax in axes:
    ax.set_ylabel("rows")
fig.tight_layout()

## The sample arithmetic — and the one number the whole study turns on

`n_eff` is **`n_dates / h`**, not `n_rows / h`. A hundred stocks on one Tuesday are
one observation of the market, not a hundred. Widening the panel adds **no
independent observations**.

What it buys is *precision per observation*: a day's IC is an average over `N`
stocks, so its sampling noise falls like `1/sqrt(N)`. That is why a cross-sectional
IC of 0.03 can mean something where a time-series IC of 0.05 did not — the daily
IC's own standard deviation is ~0.13 instead of ~1.0.

In [ ]:
labelled = panel[panel[TARGET].notna()]
dates, width = labelled["date"].nunique(), labelled.groupby("date").size().median()
arithmetic = pd.DataFrame([{
    "panel_rows": len(labelled),
    "sessions_T": dates,
    "median_width_N": int(width),
    "purge_gap_sessions": cs.PurgedWalkForwardByDate(
        horizon=HORIZON, lookback=LOOKBACK).gap,
    "n_eff_WRONG_rows_over_h": round(len(labelled) / HORIZON),
    "n_eff_dates_over_h": round(dates / HORIZON),
    "ratio": round(len(labelled) / dates, 1),
}])
arithmetic

## Run the selection

Six rankers, rank-average ensemble, correlation prune at channel level, purged
walk-forward by date. `feature_normalize="cs_rank"` replaces every feature by its
**within-date percentile** before the windowing — the representation a
cross-sectional model actually eats.

In [ ]:
def build(frame, target=TARGET, feature_normalize="cs_rank", horizon=HORIZON,
          holdout=None, repeats=10):
    return cs.CrossSectionalSelector(
        panel=frame,
        target=target,
        feature_normalize=feature_normalize,
        max_features=MAX_FEATURES,
        corr_threshold=CORR_THRESHOLD,
        horizon=horizon,
        lookback=LOOKBACK,
        n_splits=N_SPLITS,
        min_train=MIN_TRAIN,
        device=DEVICE,
        holdout_start=holdout,
        permutation_repeats=repeats,
    )


selector = build(panel)
result = selector.run(stability=False)
result.setup.to_frame().T

### The ranking

In [ ]:
result.ranking.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plots.plot_ensemble_ranking(result, top=27, ax=axes[0])
plots.plot_method_heatmap(result.scores, top=27, ax=axes[1])
fig.tight_layout()

### What was dropped, and for what

The prune is at **channel** level: the model reads a whole channel or none of it,
so redundancy is a property of channels rather than of individual summary columns.

In [ ]:
pd.Series(result.dropped_correlated, name="duplicated").to_frame()

### Signed association, and which window statistic carried each channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plots.plot_target_correlation(result.target_corr, TARGET, top=27, ax=axes[0])
plots.plot_stat_profile(result, method="xgb_shap", top=20, ax=axes[1])
fig.tight_layout()

## The walk-forward — and the error bar that matters

`result.validation` gives one mean IC per fold. That mean is an average of ~440
daily ICs whose own standard deviation is ~0.13, and `daily_ic_by_fold` reports
the distribution the mean came from, with `t = mean / (sd / sqrt(n_eff_days))`.

⚠️ **`ic_summary`'s `se_ic_per_fold` is the wrong error bar here** and is left in
only for comparability with the single-ticker tables. It is the generic
`1/sqrt(n_eff - 1)` standard error of *one* rank correlation; the fold mean is an
average of many, so its true standard error is far smaller. Read `t_stat`.

In [ ]:
result.validation.round(4)

In [ ]:
evaluation.ic_summary(result.validation, HORIZON).to_frame().T.round(4)

In [ ]:
daily = selector.daily_ic_by_fold(result)
daily

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plots.plot_validation(result.validation, ax=ax)
fig.tight_layout()

## ⚠️ The bar — the same pipeline on labels it cannot predict

The null pivots `cs_rank_5day` to a `date x ticker` matrix and permutes contiguous
blocks of `d + h` **rows**. Each stock keeps its **own** labels, moved to a
different fortnight, so cross-sectional dispersion, each name's volatility and the
label's autocorrelation all survive; only the feature-to-label pairing dies.

The **selection runs inside every draw**, because selection is the step that
inflates.

⚠️ `mode="within_date"` is the weaker null — it destroys the label's time
structure too — and is reported for contrast only.

In [ ]:
observed = evaluation.ic_summary(result.validation, HORIZON)["ic_mean"]

nulls = {}
for mode in ("date_block", "within_date"):
    nulls[mode] = cs.cross_sectional_null(
        panel, TARGET,
        factory=lambda frame: build(frame).run(stability=False),
        observed=observed, lookback=LOOKBACK, horizon=HORIZON,
        n_draws=N_NULL, seed=7, mode=mode, label=f"VN100 / {mode}",
    )

pd.DataFrame({mode: null.summary() for mode, null in nulls.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (mode, null) in zip(axes, nulls.items()):
    plots.plot_null(null, ax=ax)
fig.tight_layout()

## The holdout, with its shuffled-label control

Scored **once**. The control is the error bar: with a single number there is no
fold spread to read, and §6c showed a shuffled-label control reaching +0.169 on the
single-ticker holdout against real results of at most +0.071.

In [ ]:
hold_selector = build(panel, holdout=HOLDOUT_START)
hold_result = hold_selector.run(stability=False)
print(f"development rows {hold_result.n_rows:,}   "
      f"holdout rows {hold_selector.n_holdout:,}   "
      f"purged at the boundary {hold_selector.n_purged_at_boundary:,} "
      f"({hold_selector.cv.gap} sessions)")
hold_selector.score_holdout(hold_result).round(4)

## What moves the answer

Four one-line changes, each isolating one decision. ⚠️ Every one of them needs
**its own null** before it is compared — §8's standing rule, and `zscore` moved its
own bar from +0.053 to +0.076 in the single-ticker study without any change to the
data.

In [ ]:
variants = {
    "cs_rank feats / cs_rank_5day (baseline)": dict(),
    "RAW feats / cs_rank_5day": dict(feature_normalize="none"),
    "cs_rank feats / return_5day": dict(target="return_5day"),
    "cs_rank feats / cs_rank_10day": dict(target="cs_rank_10day", horizon=10),
}

rows = {}
for tag, kwargs in variants.items():
    horizon = kwargs.get("horizon", HORIZON)
    variant = build(panel, **kwargs)
    outcome = variant.run(stability=False)
    rows[tag] = evaluation.ic_summary(outcome.validation, horizon)
    rows[tag]["kept"] = ", ".join(outcome.kept[:4]) + " ..."
pd.DataFrame(rows).T

## ⚠️ Notes for whoever runs this next

**The GPU is not the answer here, and for a new reason.** `gpu.py` §2 measured the
GPU *losing* on a 4,235 x 27 panel because 27 columns is too little work per kernel
launch. At 250,000 x 162 that argument is gone — but `permutation_importance` is
57 % of this run and it hands the booster **host** arrays, so a `device="cuda"`
booster falls back to a DMatrix copy per prediction and the run gets slower again.
`DEVICE = "cpu"` until the permutation path keeps its data on the card.

**`permutation_repeats` is the wall-clock knob**, and the null runs use 3 rather
than 10. If you lower it, lower it for the observed run too — a null cheapened
relative to the number it is testing is a different procedure, and the procedure is
what inflates.

**Widening the universe does not lengthen the sample.** `ALL` is 7.4x the rows of
`VN100` and the same 2,862 dates, so `n_eff` is unchanged and only the per-day
precision improves. Budget accordingly: an `ALL` run is ~7x, and its null is ~7x
that again.